# College GPU reports: ogbn-arxiv, CiteSeer, Cora
Run Setup once, then run **one numbered cell at a time** (not Run All). Open the results or download the execution ZIP from the links below each cell. Full and Progressive use the same fixed dataset split and ten training seeds. Detailed kernel tracing is sampled at epoch 10 of seed 1; all recorded epochs retain wall-clock timing. On ogbn-arxiv the nested growth stages simulate graph sizes, not calendar time.

The candidate list includes the primary models and the separate PyG SGFormer. Unsupported or uninstalled models may not complete. GraphGPS uses its registered reference environment. The ogbn-arxiv recipe omits GraphGPS by default because its dense positional preprocessing can exhaust host RAM. Set INCLUDE_ARXIV_GRAPHGPS=True before running either arXiv cell only if the machine can handle it.

In [ ]:
from datetime import datetime
from pathlib import Path
from IPython.display import FileLink, display
from efficientgraphbench import Benchmark, ProgressiveBenchmark
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select the college CUDA-enabled EfficientGraphBench kernel.")

ROOT = Path("experiments") / ("college-reports-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
INCLUDE_ARXIV_GRAPHGPS = False
MODELS = ["gcn", "graphsage", "gat", "gatv2", "appnp", "sgc", "mlp",
          "sgformer_pyg", "sgformer", "graphgps", "graphormer"]

def settings(dataset, mode):
    models = [m for m in MODELS if not (dataset == "ogbn-arxiv" and m == "graphgps"
                                        and not INCLUDE_ARXIV_GRAPHGPS)]
    return dict(dataset=dataset, models=models, seeds=list(range(1, 11)), device="cuda",
                output_dir=str(ROOT / dataset / mode),
                profiling={"level": "detailed", "epochs": [10], "seeds": [1]})

def reports(dataset, mode):
    options = settings(dataset, mode)
    out = Path(options["output_dir"])
    if out.exists():
        raise FileExistsError(f"Results folder already exists: {out}. Choose a fresh ROOT to repeat.")
    if mode == "full":
        bench = Benchmark(**options, epochs=200)
    else:
        baseline = ROOT / dataset / "full"
        if not (baseline / "full-benchmark.json").is_file():
            raise FileNotFoundError(f"Run the Full cell for {dataset} first.")
        bench = ProgressiveBenchmark(**options, progressive={
            "progression_strategy": "deterministic_nested", "progression_seed": 42,
            "selection_metric": "val_f1_macro", "min_candidates": 2, "promotion_margin": 0.01,
            "full_baseline": str(baseline),
            "stages": [
                {"name": "initial", "graph_fraction": 0.25, "additional_epochs": 20, "retain_fraction": 0.7},
                {"name": "growth", "graph_fraction": 0.5, "additional_epochs": 60, "retain_fraction": 0.7},
                {"name": "full", "graph_fraction": 1.0, "additional_epochs": 120},
            ]})
    print(f"Running {dataset} / {mode} on {torch.cuda.get_device_name(0)}", flush=True)
    results = bench.run()
    perf = results.show_all(display=False)
    perf = perf["report"] if mode == "full" else perf
    execution = results.execution_analysis()
    for label, path in [("Benchmark results", perf), ("Execution analysis", execution["report"]),
                        ("Epoch and stage times", execution["data"].parent / "training-clock.csv"),
                        ("Portable execution report and traces", execution["bundle"])]:
        print(label)
        display(FileLink(str(Path(path).resolve().relative_to(Path.cwd()))))

print("GPU:", torch.cuda.get_device_name(0))
print("Report folder:", ROOT)

### 1. ogbn-arxiv — Full

In [ ]:
reports("ogbn-arxiv", "full")

### 2. ogbn-arxiv — Progressive

In [ ]:
reports("ogbn-arxiv", "progressive")

### 3. citeseer — Full

In [ ]:
reports("citeseer", "full")

### 4. citeseer — Progressive

In [ ]:
reports("citeseer", "progressive")

### 5. cora — Full

In [ ]:
reports("cora", "full")

### 6. cora — Progressive

In [ ]:
reports("cora", "progressive")